In [1]:
SYFT_VERSION = ">=0.8.2.b0,<0.9"
package_string = f'"syft{SYFT_VERSION}"'
# %pip install {package_string} -q

In [2]:
# third party
import pandas as pd

# syft absolute
import syft as sy
from syft import autocache

sy.requires(SYFT_VERSION)

✅ The installed version of syft==0.8.4b30 matches the requirement >=0.8.2b0 and the requirement <0.9


In [3]:
# Launch a fresh domain server named "test-domain-1" in dev mode on the local machine
node = sy.orchestra.launch(name="test-domain-1", port="auto", dev_mode=True, reset=True)

Staging Protocol Changes...
Starting test-domain-1 server on 0.0.0.0:54865
Waiting for server to start.

INFO:     Started server process [75389]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:54865 (Press CTRL+C to quit)


.
SQLite Store Path:
!open file:///var/folders/6_/7xx0tpq16h9cn40mq4w5gjk80000gn/T/7bca415d13ed4ec881f0d0aede098dbb.sqlite

Creating default worker image with tag='local-dev'
Building default worker image with tag=local-dev
Setting up worker poolname=default-pool workers=1 image_uid=678f2cf6657d4c3c9e07cd5865a4b655 in_memory=True
Failed to start consumer forpool=default-pool worker=default-pool-1. Error: Invalid argument (addr='tcp://localhost:None')
Failed to create container: Worker: syft.service.worker.worker_pool.SyftWorker,Error: Invalid argument (addr='tcp://localhost:None')
Data Migrated to latest version !!!
INFO:     127.0.0.1:61288 - "GET /api/v2/metadata HTTP/1.1" 200 OK
 Done.


In [9]:
client = node.login(email="info@openmined.org", password="changethis")

INFO:     127.0.0.1:63064 - "GET /api/v2/metadata HTTP/1.1" 200 OK
INFO:     127.0.0.1:63064 - "GET /api/v2/metadata HTTP/1.1" 200 OK
Logged into <test-domain-1: High-side Domain> as GUEST
INFO:     127.0.0.1:63064 - "POST /api/v2/login HTTP/1.1" 200 OK
INFO:     127.0.0.1:63064 - "GET /api/v2/api?verify_key=aec6ea4dfc049ceacaeeebc493167a88a200ddc367b1fa32da652444b635d21f&communication_protocol=dev HTTP/1.1" 200 OK
INFO:     127.0.0.1:63066 - "POST /api/v2/api_call HTTP/1.1" 200 OK
Logged into <test-domain-1: High side Domain> as <info@openmined.org>


SyftWarning: You are using a default password. Please change the password using `[your_client].me.set_password([new_password])`.

In [4]:
from cifar import get_test, get_train_batch, draw

In [15]:
train_data, train_labels = get_train_batch(1)
test_data, test_labels = get_test()

In [6]:
sy_train = sy.ActionObject.from_obj(data)
sy_labels = sy.ActionObject.from_obj(labels)

In [10]:
train_domain_obj = client.api.services.action.set(sy_train)
train_domain_labels = client.api.services.action.set(sy_labels)

INFO:     127.0.0.1:63071 - "POST /api/v2/api_call HTTP/1.1" 200 OK
INFO:     127.0.0.1:63073 - "POST /api/v2/api_call HTTP/1.1" 200 OK


In [12]:
train_domain_obj.id

<UID: 92ba2dc197bb4b739f7425e9f46d6152>

In [13]:
train_domain_label.id

<UID: 6536820f1f934b6da52104cc4cbb9695>

In [39]:
# @sy.syft_function(
#     input_policy=sy.ExactMatch(data=train_domain_obj.id, labels=train_domain_label.id),
#     output_policy=sy.SingleExecutionExactOutput(),
# )
def train_cifar10(train_data, test_data, state_dict):
    import torch
    from torch.utils.data import Dataset, DataLoader
    import torch.nn as nn
    import torchvision.transforms as transforms
    from PIL import Image
    import torch.nn.functional as F
    DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    class CustomCIFAR10Dataset(Dataset):
        def __init__(self, images, labels, transform=None):
            self.images = [img.reshape((32, 32, 3)) for img in images]
            self.labels = labels
            self.transform = transform
    
        def __len__(self):
            return len(self.images)
    
        def __getitem__(self, idx):
            image = self.images[idx]
            label = self.labels[idx]
            
            # Ensure the image is a PIL image because some transforms expect PIL images
            image = Image.fromarray(image)
    
            if self.transform:
                image = self.transform(image)
    
            return image, label

    def custom_load_data(train_data, test_data):
        transform = transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
        )
    
        trainset = CustomCIFAR10Dataset(train_data[0], train_data[1], transform=transform)
        testset = CustomCIFAR10Dataset(test_data[0], test_data[1], transform=transform)
    
        trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
        testloader = DataLoader(testset, batch_size=32)
        num_examples = {"trainset" : len(trainset), "testset" : len(testset)}
        print("num_examples", num_examples)
        return trainloader, testloader, num_examples

    def train(net, trainloader, epochs):
        """Train the network on the training set."""
        criterion = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
        for _ in range(epochs):
            for images, labels in trainloader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(net(images), labels)
                loss.backward()
                optimizer.step()
    
    
    def test(net, testloader):
        """Validate the network on the entire test set."""
        criterion = torch.nn.CrossEntropyLoss()
        correct, total, loss = 0, 0, 0.0
        with torch.no_grad():
            for data in testloader:
                images, labels = data[0].to(DEVICE), data[1].to(DEVICE)
                outputs = net(images)
                loss += criterion(outputs, labels).item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = correct / total
        return loss, accuracy
    
    class Net(nn.Module):
        def __init__(self) -> None:
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(3, 6, 5)
            self.pool = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(6, 16, 5)
            self.fc1 = nn.Linear(16 * 5 * 5, 120)
            self.fc2 = nn.Linear(120, 84)
            self.fc3 = nn.Linear(84, 10)
    
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.pool(F.relu(self.conv1(x)))
            x = self.pool(F.relu(self.conv2(x)))
            x = x.view(-1, 16 * 5 * 5)
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            x = self.fc3(x)
            return x

    net = Net().to(DEVICE)        
    trainloader, testloader, num_examples = custom_load_data(train_data, test_data)

    if state_dict:
        print("loading state dict", type(state_dict))
        net.load_state_dict(state_dict)
        loss, accuracy = test(net, testloader)
        print(f"Loss: {loss:.5f}, Accuracy:{accuracy:.3f}")
        print("ready to continue training")

    train(net, trainloader, 5)
    loss, accuracy = test(net, testloader)
    print(f"Loss: {loss:.5f}, Accuracy:{accuracy:.3f}")

    return net.state_dict()

In [37]:
state_dict = train_cifar10(
    train_data=(train_data, train_labels),
    test_data=(test_data, test_labels),
    state_dict=None
)

num_examples {'trainset': 10000, 'testset': 10000}
Loss: 614.33746, Accuracy:0.285


In [40]:
state_dict2 = train_cifar10(
    train_data=(train_data, train_labels),
    test_data=(test_data, test_labels),
    state_dict=state_dict
)

num_examples {'trainset': 10000, 'testset': 10000}
loading state dict <class 'collections.OrderedDict'>
Loss: 614.33746, Accuracy:0.285
ready to continue training
Loss: 543.22633, Accuracy:0.373
